# 🦌 Snapshot Wisconsin: Oh Deer — Multi-Task Deer Tagger

**Goal of this notebook:** from a single trail-camera image, predict two population-dynamics tags for the deer in it:

- **Antlers** — `Antlered` vs `Antlerless`
- **Age** — `Adult` vs `Young`

...plus a couple of *bonus* ecological signals (day/night, and how close/large the deer is in frame) that are cheap, consistent, and useful — worth extra points in the hackathon rubric.

### How this notebook is designed to score well

The hackathon is graded out of **150 points**:

| Bucket | Points | How this notebook targets it |
|---|---|---|
| **F1 score** (Antlers + Age) | 0–100 | One shared backbone, two heads, bbox-cropped inputs, class-weighted loss, per-task threshold tuning to maximize F1 |
| **Additional predictions** | 0–15 | Day/night (IR detection) + deer proximity/size from the bounding box — every image, cheap, ecologically meaningful |
| **Speed** | 0–15 | One forward pass predicts *both* tags; fp16 batched inference; we report images/sec |
| **Efficiency** | 0–10 | Small model (EfficientNet-B0 by default), ~5M params, runs on a single T4 |
| **Ease of use** | 0–10 | One notebook, config at the top, auto-detects the data schema, saves a reusable model + a clean `predict()` function |

> **Design philosophy:** a *useful tool*, not just a leaderboard number. Everything is configurable at the top, the code degrades gracefully if a column is named differently than expected, and the trained model is saved so it can be re-used outside this notebook.

## 1. Configuration

Everything you'd want to tweak lives here. Sensible defaults are chosen for a **single T4 GPU** on Kaggle.

In [ ]:
from types import SimpleNamespace

CFG = SimpleNamespace(
    # --- Data ---
    # Auto-detected below, but you can hard-code if needed:
    input_root   = None,          # e.g. "/kaggle/input/snapshot-wi-oh-deer"
    csv_path     = None,          # auto-globbed if None
    photos_dir   = None,          # auto-globbed if None

    # --- Model ---
    model_name   = "efficientnet_b0",   # any timm model; try "convnext_nano" for +accuracy
    img_size     = 256,
    bbox_pad     = 0.15,          # expand the bounding box by 15% before cropping
    use_bbox     = True,          # crop to the deer using the provided box

    # --- Training ---
    epochs       = 12,
    batch_size   = 32,
    lr           = 3e-4,
    weight_decay = 1e-4,
    warmup_frac  = 0.1,
    label_smooth = 0.05,
    val_frac     = 0.15,
    n_folds      = 1,             # set >1 to run stratified K-fold CV
    seed         = 42,
    num_workers  = 2,
    amp          = True,          # mixed precision (fast on T4)

    # --- Inference ---
    tune_thresholds = True,       # search decision thresholds to maximize macro-F1
    tta          = False,         # horizontal-flip TTA (small accuracy gain, 2x slower)

    # --- Output ---
    out_dir      = "/kaggle/working",
    model_file   = "oh_deer_model.pt",
    pred_file    = "predictions.csv",
)
print(CFG)

## 2. Setup & imports

`timm` gives us pretrained backbones. It's pre-installed on Kaggle's GPU images, but we install defensively.

In [ ]:
import sys, subprocess
def _pip(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

_pip("timm")

import os, glob, random, time, math, warnings
import numpy as np
import pandas as pd
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(CFG.seed)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | timm {timm.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Load data & auto-detect the schema

The dataset is **one CSV** (a row per image, 10 columns, including a bounding box) plus a folder of **~4,880 JPGs**.

Because a real-world CSV rarely has exactly the column names you expect, the cell below **auto-detects** which columns hold the filename, the Antlers label, the Age label, and the bounding box — by fuzzy keyword matching. It prints what it found so you can sanity-check. If it guesses wrong, override the `COLS` dict manually.

In [ ]:
# ---- Locate the input files (robust to Kaggle's folder layout) ----
if CFG.input_root is None:
    candidates = glob.glob("/kaggle/input/*")
    CFG.input_root = candidates[0] if candidates else "."
if CFG.csv_path is None:
    csvs = glob.glob(os.path.join(CFG.input_root, "**", "*.csv"), recursive=True)
    CFG.csv_path = csvs[0] if csvs else None
if CFG.photos_dir is None:
    # the folder that actually contains .jpg files
    jpgs = glob.glob(os.path.join(CFG.input_root, "**", "*.jpg"), recursive=True)
    CFG.photos_dir = os.path.dirname(jpgs[0]) if jpgs else CFG.input_root

print("input_root:", CFG.input_root)
print("csv_path  :", CFG.csv_path)
print("photos_dir:", CFG.photos_dir, f"({len(glob.glob(os.path.join(CFG.photos_dir, '*.jpg')))} jpgs)")

df = pd.read_csv(CFG.csv_path)
print("\nCSV shape:", df.shape)
print("Columns  :", list(df.columns))
df.head()

In [ ]:
# ---- Fuzzy-detect the columns we need ----
import re
def find_col(df, keywords, exclude=()):
    "Pick the column best matching the *earliest* (most specific) keyword.\n"
    "Short keywords (<=2 chars like 'x','w','h') match whole tokens only, so\n"
    "'h' does NOT match inside 'width'. Longer keywords match as substrings."
    def tokens(c): return set(t for t in re.split(r"[^a-z0-9]+", c.lower()) if t)
    best, best_rank = None, 1 << 30
    for c in df.columns:
        lc = c.lower()
        if any(x in lc for x in exclude):
            continue
        t = tokens(c)
        for rank, k in enumerate(keywords):
            hit = (k in t) if len(k) <= 2 else (k in lc or k in t)
            if hit and rank < best_rank:
                best, best_rank = c, rank
                break
    return best

COLS = {
    "image":   find_col(df, ["file", "image", "photo", "img", "name", "filename"]),
    "antler":  find_col(df, ["antler"]),
    # exclude image-ish words so 'age' doesn't match inside 'image_name'
    "age":     find_col(df, ["age", "young", "adult"], exclude=["image", "img"]),
    # bounding box: try both (x,y,w,h) and (xmin,ymin,xmax,ymax) conventions
    "x":       find_col(df, ["xmin", "x_min", "left", "bbox_x", "x"], exclude=["max", "flex"]),
    "y":       find_col(df, ["ymin", "y_min", "top", "bbox_y", "y"], exclude=["max"]),
    "w":       find_col(df, ["width", "w", "bbox_w"], exclude=[]),
    "h":       find_col(df, ["height", "h", "bbox_h"], exclude=[]),
    "xmax":    find_col(df, ["xmax", "x_max", "right"]),
    "ymax":    find_col(df, ["ymax", "y_max", "bottom"]),
}
print("Detected columns:")
for k, v in COLS.items():
    print(f"  {k:8s} -> {v}")

print("\n⚠️  If any label/image column is wrong, override COLS above before continuing.")
print("    e.g. COLS['antler'] = 'AntlerStatus'")

## 4. Exploratory data analysis

Two things drive the modelling choices here:

1. **Class balance.** F1 (not accuracy) is the metric precisely because these tags are imbalanced — most deer are `Adult` and, depending on season, most are `Antlerless`. A model that always predicts the majority class gets high accuracy but terrible F1. We'll counter this with **class-weighted loss** and **threshold tuning**.
2. **Bounding-box coverage.** Antlers are a *fine* detail. Cropping to the deer (using the provided box) removes distracting background and makes the antler signal far easier to learn.

In [ ]:
def normalize_label(series, positive_keys, negative_keys):
    "Map a messy text/numeric column to {0,1} using keyword lists."
    s = series.astype(str).str.strip().str.lower()
    out = pd.Series(index=s.index, dtype="float")
    for i, v in s.items():
        if any(k in v for k in positive_keys):
            out[i] = 1.0
        elif any(k in v for k in negative_keys):
            out[i] = 0.0
        else:
            out[i] = np.nan
    return out

# Class 1 = the "interesting"/rarer class in each task
df["y_antler"] = normalize_label(df[COLS["antler"]], ["antlered", "yes", "1", "true", "buck"], ["antlerless", "no", "0", "false", "doe"])
df["y_age"]    = normalize_label(df[COLS["age"]],    ["young", "fawn", "juvenile", "1"],        ["adult", "mature", "0"])

print("Antlers  (1=Antlered):")
print(df["y_antler"].value_counts(dropna=False), "\n")
print("Age      (1=Young):")
print(df["y_age"].value_counts(dropna=False))

n_bad = df["y_antler"].isna().sum() + df["y_age"].isna().sum()
if n_bad:
    print(f"\n⚠️  {n_bad} label cells couldn't be mapped — inspect the raw values below and "
          f"adjust the keyword lists in normalize_label().")
    print("Raw Antlers values:", df[COLS['antler']].astype(str).str.lower().value_counts().head(10).to_dict())
    print("Raw Age values     :", df[COLS['age']].astype(str).str.lower().value_counts().head(10).to_dict())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
df["y_antler"].map({0:"Antlerless",1:"Antlered"}).value_counts().plot.bar(ax=ax[0], color="#4C78A8", title="Antlers")
df["y_age"].map({0:"Adult",1:"Young"}).value_counts().plot.bar(ax=ax[1], color="#F58518", title="Age")
# joint distribution
joint = df.groupby(["y_antler","y_age"]).size().rename("n").reset_index()
ax[2].bar([f"A{int(a)}/Ag{int(g)}" for a,g in zip(joint.y_antler, joint.y_age)], joint.n, color="#54A24B")
ax[2].set_title("Joint (Antler/Age)")
for a in ax: a.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

# Quick day/night probe on a sample (color saturation ~ 0 for IR night shots)
def is_night(path, thresh=12):
    try:
        im = np.asarray(Image.open(path).convert("RGB").resize((64,64)), dtype=np.float32)
        sat = (im.max(-1) - im.min(-1)).mean()   # mean chroma
        return sat < thresh
    except Exception:
        return None

sample = df[COLS["image"]].dropna().sample(min(200, len(df)), random_state=0)
def resolve(fn): return fn if os.path.isabs(str(fn)) else os.path.join(CFG.photos_dir, str(fn))
night_frac = np.mean([is_night(resolve(f)) for f in sample])
print(f"\nEstimated night (IR) fraction in a 200-image sample: {night_frac:.0%}")

## 5. Dataset, bounding-box cropping & augmentation

The `DeerDataset` reads an image, optionally **crops to the deer** using the bounding box (expanded by `bbox_pad` so we don't clip antlers), and applies augmentation. Both bbox conventions — `(x, y, w, h)` and `(xmin, ymin, xmax, ymax)` — and both normalized (0–1) and pixel coordinates are handled.

Augmentation is deliberately mild and *label-preserving*: horizontal flip (a flipped buck is still a buck), small rotation/scale/color jitter to simulate different cameras, lighting, and seasons.

In [ ]:
import torchvision.transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_bbox(row, W, H):
    "Return (left, top, right, bottom) in pixels, or None if unavailable."
    c = COLS
    try:
        if c["x"] and c["y"] and c["w"] and c["h"] and c["xmax"] is None:
            x, y, w, h = [float(row[c[k]]) for k in ("x","y","w","h")]
            l, t, r, b = x, y, x+w, y+h
        elif c["x"] and c["y"] and c["xmax"] and c["ymax"]:
            l, t, r, b = [float(row[c[k]]) for k in ("x","y","xmax","ymax")]
        else:
            return None
        # Normalized coords? scale up.
        if max(l, t, r, b) <= 1.5:
            l, r = l*W, r*W; t, b = t*H, b*H
        if not (r > l and b > t):
            return None
        return l, t, r, b
    except (ValueError, TypeError, KeyError):
        return None

def crop_to_bbox(img, box, pad):
    W, H = img.size
    l, t, r, b = box
    pw, ph = (r-l)*pad, (b-t)*pad
    l, t, r, b = max(0, l-pw), max(0, t-ph), min(W, r+pw), min(H, b+ph)
    return img.crop((int(l), int(t), int(r), int(b)))

class DeerDataset(Dataset):
    def __init__(self, frame, train=False):
        self.df = frame.reset_index(drop=True)
        self.train = train
        aug = [T.Resize((CFG.img_size, CFG.img_size))]
        if train:
            aug = [
                T.RandomResizedCrop(CFG.img_size, scale=(0.75, 1.0), ratio=(0.8, 1.25)),
                T.RandomHorizontalFlip(),
                T.RandomRotation(8),
                T.ColorJitter(0.2, 0.2, 0.2, 0.02),
            ]
        self.tf = T.Compose(aug + [T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        path = resolve(row[COLS["image"]])
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (CFG.img_size, CFG.img_size))
        if CFG.use_bbox:
            box = get_bbox(row, *img.size)
            if box: img = crop_to_bbox(img, box, CFG.bbox_pad)
        x = self.tf(img)
        ya = torch.tensor(row["y_antler"], dtype=torch.long)
        yg = torch.tensor(row["y_age"], dtype=torch.long)
        return x, ya, yg

## 6. Multi-task model

**One backbone, two heads.** The shared EfficientNet extracts features once; two tiny linear heads read off `Antlers` and `Age`. This is the crux of the *speed* and *efficiency* story — you get both predictions from a **single forward pass**, roughly halving inference cost versus two separate models, with fewer parameters to train and less to overfit on a small dataset.

In [ ]:
class MultiTaskDeer(nn.Module):
    def __init__(self, backbone=CFG.model_name, n_antler=2, n_age=2, drop=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=True, num_classes=0, global_pool="avg")
        feat = self.backbone.num_features
        self.drop = nn.Dropout(drop)
        self.head_antler = nn.Linear(feat, n_antler)
        self.head_age    = nn.Linear(feat, n_age)

    def forward(self, x):
        f = self.drop(self.backbone(x))
        return self.head_antler(f), self.head_age(f)

_m = MultiTaskDeer()
n_params = sum(p.numel() for p in _m.parameters()) / 1e6
print(f"{CFG.model_name}: {n_params:.1f}M parameters")
del _m

## 7. Train

Key choices, and *why*:

- **Class-weighted cross-entropy** — weights the rarer class (Antlered / Young) up, directly targeting F1 rather than accuracy.
- **Cosine LR with warmup** + **mixed precision (AMP)** — fast, stable convergence on a T4.
- **Model selection on validation macro-F1** (the mean of the two tasks' macro-F1), *not* on loss — because that's what the leaderboard rewards.
- Drop any rows whose labels couldn't be parsed, so training never sees `NaN` targets.

In [ ]:
def make_loaders(tr, va):
    dl_tr = DataLoader(DeerDataset(tr, train=True),  batch_size=CFG.batch_size, shuffle=True,
                       num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
    dl_va = DataLoader(DeerDataset(va, train=False), batch_size=CFG.batch_size*2, shuffle=False,
                       num_workers=CFG.num_workers, pin_memory=True)
    return dl_tr, dl_va

def class_weights(y):
    vals, counts = np.unique(y, return_counts=True)
    w = counts.sum() / (len(vals) * counts)
    out = torch.ones(2)
    for v, wi in zip(vals, w): out[int(v)] = wi
    return out.float().to(DEVICE)

@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    pa, pg, ta, tg = [], [], [], []
    proba_a, proba_g = [], []
    for x, ya, yg in dl:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE, enabled=CFG.amp):
            la, lg = model(x)
        proba_a.append(la.softmax(1)[:,1].float().cpu().numpy())
        proba_g.append(lg.softmax(1)[:,1].float().cpu().numpy())
        ta.append(ya.numpy()); tg.append(yg.numpy())
    proba_a = np.concatenate(proba_a); proba_g = np.concatenate(proba_g)
    ta = np.concatenate(ta); tg = np.concatenate(tg)
    fa = f1_score(ta, (proba_a>0.5).astype(int), average="macro")
    fg = f1_score(tg, (proba_g>0.5).astype(int), average="macro")
    return (fa+fg)/2, fa, fg, (proba_a, proba_g, ta, tg)

def train_one(tr, va, tag="model"):
    dl_tr, dl_va = make_loaders(tr, va)
    model = MultiTaskDeer().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    steps = len(dl_tr) * CFG.epochs
    warm = int(steps * CFG.warmup_frac)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s/max(1,warm) if s < warm else 0.5*(1+math.cos(math.pi*(s-warm)/max(1,steps-warm))))
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.amp)
    w_a = class_weights(tr["y_antler"].values); w_g = class_weights(tr["y_age"].values)
    ce_a = nn.CrossEntropyLoss(weight=w_a, label_smoothing=CFG.label_smooth)
    ce_g = nn.CrossEntropyLoss(weight=w_g, label_smoothing=CFG.label_smooth)

    best, best_state = -1, None
    for ep in range(CFG.epochs):
        model.train(); t0 = time.time(); running = 0.0
        for x, ya, yg in dl_tr:
            x, ya, yg = x.to(DEVICE), ya.to(DEVICE), yg.to(DEVICE)
            opt.zero_grad()
            with torch.autocast(DEVICE, enabled=CFG.amp):
                la, lg = model(x)
                loss = ce_a(la, ya) + ce_g(lg, yg)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            running += loss.item()
        mf1, fa, fg, _ = evaluate(model, dl_va)
        print(f"[{tag}] epoch {ep+1:02d}/{CFG.epochs}  loss {running/len(dl_tr):.3f}  "
              f"macroF1 {mf1:.4f} (antler {fa:.3f} | age {fg:.3f})  {time.time()-t0:.0f}s")
        if mf1 > best:
            best, best_state = mf1, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f"[{tag}] best val macro-F1: {best:.4f}")
    return model

# Drop unparseable labels, then split (stratified on the joint class)
data = df.dropna(subset=["y_antler", "y_age"]).reset_index(drop=True)
data["y_antler"] = data["y_antler"].astype(int); data["y_age"] = data["y_age"].astype(int)
data["_strat"] = data["y_antler"].astype(str) + data["y_age"].astype(str)
print(f"Training on {len(data)} labelled images "
      f"({len(df)-len(data)} dropped for unparseable labels)")

tr, va = train_test_split(data, test_size=CFG.val_frac, random_state=CFG.seed, stratify=data["_strat"])
model = train_one(tr, va, tag="main")

## 8. Threshold tuning & final validation report

With imbalanced classes, `0.5` is rarely the F1-optimal cutoff. We sweep thresholds on the validation set for each task and lock in the ones that maximize **macro-F1**. This is a legitimate, standard step — thresholds are tuned on *validation*, never on test.

In [ ]:
mf1, fa, fg, (pa, pg, ta, tg) = evaluate(model, make_loaders(tr, va)[1])

def best_threshold(proba, y):
    grid = np.linspace(0.1, 0.9, 81)
    scores = [f1_score(y, (proba>t).astype(int), average="macro") for t in grid]
    j = int(np.argmax(scores)); return float(grid[j]), float(scores[j])

if CFG.tune_thresholds:
    th_a, f_a = best_threshold(pa, ta)
    th_g, f_g = best_threshold(pg, tg)
else:
    th_a, th_g = 0.5, 0.5
    f_a = f1_score(ta, (pa>0.5).astype(int), average="macro")
    f_g = f1_score(tg, (pg>0.5).astype(int), average="macro")

CFG.threshold_antler, CFG.threshold_age = th_a, th_g
print(f"Antlers  threshold={th_a:.2f}  macro-F1={f_a:.4f}")
print(f"Age      threshold={th_g:.2f}  macro-F1={f_g:.4f}")
print(f"\n>>> Combined leaderboard proxy (mean macro-F1): {(f_a+f_g)/2:.4f}"
      f"  ->  ~{(f_a+f_g)/2*100:.1f} / 100 points\n")

print("=== Antlers ===")
print(classification_report(ta, (pa>th_a).astype(int), target_names=["Antlerless","Antlered"], digits=3))
print("=== Age ===")
print(classification_report(tg, (pg>th_g).astype(int), target_names=["Adult","Young"], digits=3))

## 9. Bonus predictions (the "additional predictions" points)

The rubric rewards *extra* ecological signals that are (a) predicted for **every** image and (b) genuinely useful. Two cheap, robust ones:

1. **Time of day — Day vs Night.** Trail cameras switch to **infrared** at night, producing (near-)grayscale frames. Mean chroma (color saturation) cleanly separates day from IR-night. This is directly useful for studying deer **activity patterns** (crepuscular behavior).
2. **Proximity / relative size.** The fraction of the frame the deer occupies (from the bounding box) is a proxy for **distance from the camera** — useful for detection-range calibration and density estimation.

Both are deterministic, instant, and require no extra model — a strong efficiency story.

In [ ]:
def day_night(path, thresh=12):
    try:
        im = np.asarray(Image.open(path).convert("RGB").resize((64,64)), dtype=np.float32)
        sat = float((im.max(-1) - im.min(-1)).mean())
        return ("Night" if sat < thresh else "Day"), sat
    except Exception:
        return ("Unknown", np.nan)

def proximity(row):
    "Bbox area as a fraction of the frame -> Close / Mid / Far."
    try:
        img = Image.open(resolve(row[COLS["image"]]))
        box = get_bbox(row, *img.size)
        if not box: return ("Unknown", np.nan)
        l,t,r,b = box; frac = ((r-l)*(b-t)) / (img.size[0]*img.size[1])
        lvl = "Close" if frac > 0.25 else ("Mid" if frac > 0.06 else "Far")
        return (lvl, round(float(frac),4))
    except Exception:
        return ("Unknown", np.nan)

# demo on a few rows
for _, r in data.sample(5, random_state=1).iterrows():
    dn, sat = day_night(resolve(r[COLS["image"]]))
    px, frac = proximity(r)
    print(f"{str(r[COLS['image']])[:40]:40s}  {dn:5s} (sat {sat:5.1f})  {px:6s} (area {frac})")

## 10. Fast batched inference + the prediction file

A single `predict_frame()` runs the whole tool over any dataframe of images: bbox crop → one forward pass for both tags → bonus signals. We time it and report **images/sec** (the speed metric), then write a tidy `predictions.csv`.

> **Note on submission format:** the hackathon's exact prediction-file spec lives in Rules §4/§6 (which don't render cleanly on the site). This CSV carries the image id + both required predictions in human-readable form; adjust the column names in the last cell to match the official template once you've read it.

In [ ]:
@torch.no_grad()
def predict_frame(frame, with_bonus=True):
    ds = DeerDataset(frame.assign(y_antler=0, y_age=0), train=False)
    dl = DataLoader(ds, batch_size=CFG.batch_size*2, shuffle=False, num_workers=CFG.num_workers)
    model.eval(); pa, pg = [], []
    t0 = time.time()
    for x, _, _ in dl:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE, enabled=CFG.amp):
            la, lg = model(x)
            if CFG.tta:
                la2, lg2 = model(torch.flip(x, dims=[3]))
                la = (la.softmax(1)+la2.softmax(1))/2; lg = (lg.softmax(1)+lg2.softmax(1))/2
            else:
                la, lg = la.softmax(1), lg.softmax(1)
        pa.append(la[:,1].float().cpu().numpy()); pg.append(lg[:,1].float().cpu().numpy())
    dt = time.time()-t0
    pa = np.concatenate(pa); pg = np.concatenate(pg)
    print(f"Inference: {len(frame)} images in {dt:.1f}s  ->  {len(frame)/dt:.1f} images/sec "
          f"({DEVICE}, {CFG.model_name}, fp16={CFG.amp})")

    out = pd.DataFrame({COLS["image"]: frame[COLS["image"]].values})
    out["Antlers"] = np.where(pa > CFG.threshold_antler, "Antlered", "Antlerless")
    out["Age"]     = np.where(pg > CFG.threshold_age, "Young", "Adult")
    out["antler_prob"] = pa.round(4); out["age_prob"] = pg.round(4)
    if with_bonus:
        dn = [day_night(resolve(f)) for f in frame[COLS["image"]]]
        out["time_of_day"] = [d[0] for d in dn]
        px = [proximity(r) for _, r in frame.iterrows()]
        out["proximity"] = [p[0] for p in px]
        out["bbox_area_frac"] = [p[1] for p in px]
    return out

preds = predict_frame(data)
preds.to_csv(os.path.join(CFG.out_dir, CFG.pred_file), index=False)
print("Saved:", os.path.join(CFG.out_dir, CFG.pred_file))
preds.head(10)

## 11. Save the model (reusability = ease-of-use points)

We save weights **plus** the config and tuned thresholds, so the tool can be reloaded and used on new trail-camera images with a few lines — no retraining. Include this snippet in your GitHub README.

In [ ]:
ckpt = {
    "state_dict": model.state_dict(),
    "model_name": CFG.model_name,
    "img_size":   CFG.img_size,
    "threshold_antler": CFG.threshold_antler,
    "threshold_age":    CFG.threshold_age,
    "cols": COLS,
}
torch.save(ckpt, os.path.join(CFG.out_dir, CFG.model_file))
print("Saved model ->", os.path.join(CFG.out_dir, CFG.model_file))

# ---- Minimal reuse example (works standalone) ----
print(r'''
# --- Reuse on new images ---
ckpt = torch.load("oh_deer_model.pt", map_location="cpu")
net = MultiTaskDeer(backbone=ckpt["model_name"]).eval()
net.load_state_dict(ckpt["state_dict"])
tf = T.Compose([T.Resize((ckpt["img_size"],)*2), T.ToTensor(),
                T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
img = Image.open("some_deer.jpg").convert("RGB")
with torch.no_grad():
    la, lg = net(tf(img).unsqueeze(0))
antler = "Antlered" if la.softmax(1)[0,1] > ckpt["threshold_antler"] else "Antlerless"
age    = "Young"    if lg.softmax(1)[0,1] > ckpt["threshold_age"]    else "Adult"
print(antler, age)
''')

## 12. Where to push for a higher score

Concrete, ranked next steps (each is a small edit above):

1. **Bigger backbone if F1 stalls** — set `CFG.model_name = "convnext_tiny"` or `"tf_efficientnetv2_s"`. Costs speed/efficiency points, so only if F1 gains outweigh them.
2. **K-fold CV + ensembling** — set `CFG.n_folds = 5`, average the fold probabilities. Usually +1–3 F1 points; keep one fold's model for the "fast" story.
3. **Enable TTA** (`CFG.tta = True`) for a small, cheap accuracy bump at 2× inference cost.
4. **A third bonus prediction** — e.g. **deer count** via a pretrained camera-trap detector like **MegaDetector**; very credible with these ecology judges. Adds compute, so gate it behind a flag.
5. **Error review** — plot the worst false positives/negatives (especially Young/fawn, the rarest class). Often reveals label noise or bbox issues you can fix.

Remember the meta-game: this is judged on **usefulness**, not just F1. A clean repo, a clear README, honest error analysis, and fast/efficient inference are worth **50 of the 150 points** — don't leave them on the table.